In [ ]:
import sys, os; sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), '..')))
#import os; os.chdir(os.path.dirname(os.getcwd()))
from utils.model_loader import get_model_fits
import numpy as np
import pandas as pd
import re
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
data_dir = "datasets/abalone"
results_dir_tanh = "results/regression/single_layer/tanh/abalone"

full_config_path = "abalone_N3341_p8"
tanh_fit = get_model_fits(
    config=full_config_path,
    results_dir=results_dir_tanh,
    include_prior=False,
)


In [ ]:
from sklearn.metrics import mean_squared_error
from properscoring import crps_ensemble
import numpy as np
import pandas as pd

from utils.generate_data import load_abalone_regression_data
X_train, X_test, y_train, y_test = load_abalone_regression_data(standardized=False, frac=1.0)

rows = []
for model_name, model_entry in tanh_fit.items():
    post = model_entry["posterior"]

    y_samps = post.stan_variable("output_test").squeeze(-1)

    y_mean = y_samps.mean(axis=0)                                   # (n_test,)
    rmse_post_mean = float(np.sqrt(mean_squared_error(y_test, y_mean)))

    per_draw_rmse = np.sqrt(((y_samps - y_test[None, :])**2).mean(axis=1))  # (S,)
    rmse_draw_mean = float(per_draw_rmse.mean())

    crps = float(np.mean(crps_ensemble(y_test, y_samps.T)))

    rows.append({
        "Model": model_name,
        "RMSE_posterior_mean": rmse_post_mean,
        "RMSE_mean_over_draws": rmse_draw_mean,
        "CRPS": crps,
        "n_draws": y_samps.shape[0]
    })

results_df = pd.DataFrame(rows).sort_values("RMSE_posterior_mean")
print(results_df)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from utils.generate_data import load_abalone_regression_data
from utils.sparsity import forward_pass_relu, forward_pass_tanh, local_prune_weights

def evaluate_posterior_on_multiple_testsets(
    fits,
    models,
    frac,
    forward_pass,
    n_testsets=5,
):
    rows = []

    for test_id in range(n_testsets):
        _, X_test, _, y_test = load_abalone_regression_data(
            standardized=False,
            frac=frac,
            random_state=42 + test_id
        )

        X_test_np = X_test.to_numpy()
        y_test_np = y_test.reshape(-1)

        for model in models:
            fit = fits[model]["posterior"]

            W1_samples = fit.stan_variable("W_1")
            W2_samples = fit.stan_variable("W_L")
            b1_samples = fit.stan_variable("hidden_bias")
            b2_samples = fit.stan_variable("output_bias")

            S = W1_samples.shape[0]
            y_hats = np.zeros((S, y_test_np.shape[0]))
            rmse = np.zeros((S))

            for i in range(S):
                y_hat = forward_pass(
                    X_test_np,
                    W1_samples[i],
                    np.asarray(b1_samples[i]).reshape(-1),
                    W2_samples[i],
                    np.asarray(b2_samples[i]).reshape(-1),
                )
                y_hats[i] = y_hat.squeeze()
                #rmse[i] = np.sqrt(mean_squared_error(y_test_np, y_hats[i]))
                
            
            y_mean = y_hats.mean(axis=0)
            posterior_rmse = np.sqrt(mean_squared_error(y_test_np, y_mean))

            rows.append({
                "model": model,
                "test_set": test_id,
                "posterior_rmse": posterior_rmse,
                #"mean_rmse": rmse.mean(axis=0)
            })

    df = pd.DataFrame(rows)

    # 🔹 THIS is the only new part
    df_mean = (
        df.groupby("model", as_index=False)["posterior_rmse"]
          .mean()
          .rename(columns={"posterior_rmse": "mean_rmse_over_testsets"})
    )

    return df_mean, df


In [ ]:
models = list(tanh_fit.keys())  # e.g. ["Gaussian", "Regularized Horseshoe", ...]
df_results, df = evaluate_posterior_on_multiple_testsets(
    fits=tanh_fit,
    models=models,
    frac=0.5,        # YOU control this
    forward_pass=forward_pass_tanh,
    n_testsets=5,
)

print(df_results)

In [ ]:
latex_table = results_df.to_latex(index=False, float_format="%.4f", column_format="lcc", caption="RMSE and CRPS per model.", label="tab:rmse_crps")
print(latex_table)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from utils.generate_data import load_abalone_regression_data
X_train, _, _, _ = load_abalone_regression_data(standardized=False, frac=1.0)

P = 8
H = 16
L = 1
out_nodes = 1

layer_structure = {
    'input_to_hidden': {'name': 'W_1', 'shape': (P, H)},
    'hidden_to_output': {'name': 'W_L', 'shape': (H, out_nodes)}
}


def build_single_draw_weights(fits, layer_structure, draw_idx):
    """Return {model: {'W_1': (P,H), 'W_L': (H,O)}} for ONE draw."""
    out = {}
    for name, fd in fits.items():
        fit = fd["posterior"]
        W1 = fit.stan_variable(layer_structure['input_to_hidden']['name'])[draw_idx]
        WL = fit.stan_variable(layer_structure['hidden_to_output']['name'])[draw_idx]
        WL = WL.reshape(layer_structure['hidden_to_output']['shape'])
        out[name] = {"W_1": W1, "W_L": WL}
    return out

def scale_W1_for_plot(model_means, mode='global'):
    """
    Skalerer alle W_1 til [-1, 1] for rettferdig sammenligning av edge-tykkelser.

    mode:
      - 'global' : én felles skala over alle modeller (mest sammenlignbar)
      - 'per_model': egen skala per modell (uavhengig sammenligning)
      - 'per_node' : skalerer hver kolonne (node) separat til [-1,1]

    Returnerer: scaled_model_means (samme struktur som input), scale_info
    """
    scaled = {}
    if mode == 'global':
        gmax = max(np.abs(m['W_1']).max() for m in model_means.values())
        gmax = max(gmax, 1e-12)
        for name, m in model_means.items():
            W1s = m['W_1'] / gmax
            out = {k: v for k, v in m.items()}
            out['W_1'] = W1s
            scaled[name] = out
        return scaled, {'mode': 'global', 'scale': gmax}

    elif mode == 'per_model':
        for name, m in model_means.items():
            s = max(np.abs(m['W_1']).max(), 1e-12)
            out = {k: v for k, v in m.items()}
            out['W_1'] = m['W_1'] / s
            scaled[name] = out
        return scaled, {'mode': 'per_model'}

    elif mode == 'per_node':
        for name, m in model_means.items():
            W1 = m['W_1'].copy()
            P, H = W1.shape
            for h in range(H):
                colmax = max(np.abs(W1[:, h]).max(), 1e-12)
                W1[:, h] = W1[:, h] / colmax
            out = {k: v for k, v in m.items()}
            out['W_1'] = W1
            scaled[name] = out
        return scaled, {'mode': 'per_node'}

    else:
        raise ValueError("mode must be 'global', 'per_model', or 'per_node'")
feature_names = list(X_train.columns)

abbr = {
    "Gaussian": "Gauss",
    "RHS": "RHS",
    "DHS": "DHS",
    "DST": "DST",
    "Beta Horseshoe tanh": "BHS",
    "Beta Student T tanh": "BST",

}

def plot_models_with_activations(model_means, layer_sizes,
                                 activations=None, activation_color_max=None,
                                 ncols=3, figsize_per_plot=(5,4), signed_colors=False, feature_names=None):
    """
    model_means: dict {model_name: {'W_1':(P,H), 'W_L':(H,O), optional 'W_internal':[...]} }
    layer_sizes: f.eks [P, H, O] eller [P, H, H, O] ved internlag
    activations: dict {model_name: (H,)} – aktiveringsfrekvens kun for første skjulte lag
    activation_color_max: global maks for skalering av farger (hvis None brukes 1.0)
    """
    names = list(model_means.keys())
    n_models = len(names)
    nrows = int(np.ceil(n_models / ncols))
    figsize = (figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows * ncols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    # Skru av blanke akser
    for ax in axes[n_models:]:
        ax.axis('off')

    for ax, name in zip(axes, names):
        weights = model_means[name]
        G = nx.DiGraph()
        pos, nodes_per_layer, node_colors = {}, [], []

        # Noder med posisjon og farge
        for li, size in enumerate(layer_sizes):
            ids = []
            ycoords = np.linspace(size - 1, 0, size) - (size - 1) / 2
            for i in range(size):
                nid = f"L{li}_{i}"
                G.add_node(nid)
                pos[nid] = (li, ycoords[i])
                ids.append(nid)
                if li == 0 and feature_names is not None:
                    ax.text(pos[nid][0]-0.12, pos[nid][1], feature_names[i],
                            ha='right', va='center', fontsize=8)

                if activations is not None and li == 1:  # kun første skjulte lag
                    #a = activations.get(name, np.zeros(size))
                    a = activations.get(name, np.zeros(size))
                    a = np.asarray(a).ravel()   # <-- flater til 1D array
                    scale = activation_color_max if activation_color_max is not None else 1.0
                    val = float(np.clip(a[i] / max(scale, 1e-12), 0.0, 1.0))
                    color = plt.cm.winter(val)
                else:
                    color = 'lightblue'
                node_colors.append(color)

            nodes_per_layer.append(ids)

        edge_colors, edge_widths = [], []

        def add_edges(W, inn, ut):
            for j, out_n in enumerate(ut):
                for i, in_n in enumerate(inn):
                    w = float(W[i, j])
                    G.add_edge(in_n, out_n, weight=abs(w))
                    edge_colors.append('red' if w >= 0 else 'blue')
                    edge_widths.append(abs(w))

        # input -> hidden(1)
        add_edges(weights['W_1'], nodes_per_layer[0], nodes_per_layer[1])

        # ev. internlag
        if 'W_internal' in weights:
            for l, Win in enumerate(weights['W_internal']):
                add_edges(Win, nodes_per_layer[l+1], nodes_per_layer[l+2])

        # siste hidden -> output
        add_edges(weights['W_L'], nodes_per_layer[-2], nodes_per_layer[-1])

        nx.draw(G, pos, ax=ax,
                node_color=node_colors,
                edge_color=(edge_colors if signed_colors else 'red'),
                width=[G[u][v]['weight'] for u,v in G.edges()],
                with_labels=False, node_size=400, arrows=False)

        ax.set_title(abbr[name], fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    return fig

def compute_hidden_activation(fit_dict, x_train, draw_idx):
    fit = fit_dict['posterior']
    W1 = fit.stan_variable('W_1')[draw_idx, :, :]          # (P,H)
    try:
        b1 = fit.stan_variable('hidden_bias')[draw_idx, :] # (H,)
    except Exception:
        b1 = np.zeros(W1.shape[1])
    # tanh i [-1,1]
    a_full = np.tanh(x_train @ W1 + b1)             # (H,)
    a=np.mean(a_full, axis=0)
    return a


In [ ]:
# Velg en observasjon å "lyse opp" nodefargene med
obs_idx = 3
draw_idx = 69 #pick_draw_idx(prior_fits, seed=42)      # one common draw across models
prior_draws = build_single_draw_weights(tanh_fit, layer_structure, draw_idx)

# 1) Beregn aktivasjoner for ALLE modellene
activations = {}
for name, fd in tanh_fit.items():
    a = compute_hidden_activation(fd, X_train, draw_idx)
    activations[name] = np.abs(a)      

# 2) Skaler vekter for plotting (som før)
scaled, _ = scale_W1_for_plot(prior_draws, mode='per_model')

# 3) Kall plottet med aktivasjoner
# Siden tanh ∈ [-1,1] og vi bruker |a|, så sett activation_color_max=1.0
fig = plot_models_with_activations(
    scaled,
    layer_sizes=[P, H, out_nodes],
    activations=None,
    activation_color_max=1.0,
    ncols=2,
    feature_names = None
)
plt.savefig("figures_for_use_in_paper/abalone_network_tanh_with_beta.png", bbox_inches="tight")
plt.show()

In [ ]:
from utils.generate_data import load_abalone_regression_data
X_train, X_test, y_train, y_test = load_abalone_regression_data(standardized=False, frac=1.0)
print(X_train.shape, X_test.shape)

In [ ]:
from utils.robust_utils import build_pytorch_model_from_stan_sample
import torch
import numpy as np
import shap

fit = tanh_fit['DST']['posterior']

P = X_train.shape[1]
H = 16

# 1) Build the torch model from one Stan draw
model = build_pytorch_model_from_stan_sample(
    fit, sample_idx=69, input_dim=P, hidden_dim=H,
    output_dim=1, task="regression", activation=torch.tanh
)
model.eval()

# 2) Predict function
def predict_numpy(X_np):
    with torch.no_grad():
        X_t = torch.tensor(X_np, dtype=torch.float32)
        y = model(X_t).cpu().numpy()
    return y

# 3) Background and eval sets for KernelSHAP
feature_names = list(X_train.columns)
X_bg   = X_train.sample(n=200, random_state=0).to_numpy(dtype=float)
X_eval = X_test.sample(n=200, random_state=1).to_numpy(dtype=float)

# 4) KernelSHAP
explainer = shap.KernelExplainer(predict_numpy, X_bg)
shap_vals = explainer.shap_values(X_eval)

# 5) Global importance (mean |SHAP|)
mean_abs = np.abs(shap_vals).mean(axis=0)
order = np.argsort(mean_abs)[::-1]
for j in order:
    print(f"{feature_names[j]:16s}  {mean_abs[j]:.4f}")

shap.summary_plot(shap_vals, X_eval, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
corr = pd.DataFrame(X_train, columns=X_train.columns).drop(columns=["Sex"]).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0.92)
plt.show()
